# 13.01 Anthropic — 공급자 카탈로그

**`langchain-anthropic`** 한 패키지로 Claude 계열 chat 모델과 **Claude 전용 미들웨어 5종**(prompt caching · bash · text editor · memory · file search) 을 모두 끌고 온다.
이 노트북은 **공급자 관점 오버뷰**다 — 카테고리별 깊은 사용은 각 카테고리 노트북으로 점프한다.

## 학습 목표

- Anthropic 패키지가 다루는 **integration 전체 지도** 파악
- Claude **모델 ID 명명 규칙** (Sonnet 4.5/4.6 · Opus 4.7 · Haiku 3.5/4)
- 공급자 특화 기능 1순위: **prompt caching** 으로 입력 토큰 90% 절감
- 가격 · 한도 · 리전 빠른 참조표

## 언제 쓰나

- 추론 품질 + 긴 컨텍스트(200k~1M) 가 필요한 에이전트
- 도구 호출 정확도가 비용보다 우선인 코드/리뷰 에이전트
- prompt caching 으로 **system prompt 재사용** 비중이 큰 워크로드

## 13.01.1 환경 설정

필요 패키지: `langchain-anthropic`. `.env` 에 `ANTHROPIC_API_KEY` 가 있어야 한다.

```bash
uv pip install langchain-anthropic
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ["ANTHROPIC_API_KEY"], "ANTHROPIC_API_KEY 필요"

## 13.01.2 공급자 제품군 전체 지도

한 패키지에서 끌어오는 모든 심볼을 카테고리별로 정리한다.

| 카테고리 | 심볼 | 카테고리 노트북 |
|---------|------|----------------|
| Chat | `ChatAnthropic` | [`01_chat_models/02_anthropic.ipynb`](../01_chat_models/02_anthropic.ipynb) |
| Middleware (caching) | `AnthropicPromptCachingMiddleware` | [`11_provider_middleware/01_anthropic_prompt_caching.ipynb`](../11_provider_middleware/01_anthropic_prompt_caching.ipynb) |
| Middleware (bash 도구) | `ClaudeBashToolMiddleware` | [`11_provider_middleware/02_claude_bash_tool.ipynb`](../11_provider_middleware/02_claude_bash_tool.ipynb) |
| Middleware (text editor) | `ClaudeTextEditorToolMiddleware` | [`11_provider_middleware/03_claude_text_editor.ipynb`](../11_provider_middleware/03_claude_text_editor.ipynb) |
| Middleware (memory) | `ClaudeMemoryToolMiddleware` | [`11_provider_middleware/04_claude_memory.ipynb`](../11_provider_middleware/04_claude_memory.ipynb) |
| Middleware (file search) | `AnthropicFileSearchMiddleware` | [`11_provider_middleware/05_anthropic_file_search.ipynb`](../11_provider_middleware/05_anthropic_file_search.ipynb) |
| Bedrock 경유 Claude | `ChatBedrockConverse` (`langchain-aws`) | [`01_chat_models/05_bedrock.ipynb`](../01_chat_models/05_bedrock.ipynb) |

Anthropic 은 **embedding · vector store · retriever 를 자체 제공하지 않는다** — 임베딩은 OpenAI/Voyage/Cohere/Google 중 하나를 짝지어 쓴다.

## 13.01.3 기본 사용 — chat

최소 한 줄. `init_chat_model("anthropic:...")` 또는 직접 `ChatAnthropic(...)` 둘 다 가능하다.

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=256)
print(model.invoke("한국어로 한 문장만: 너는 누구냐?").content)

## 13.01.4 공급자 특화 기능 — prompt caching

Anthropic 만 가진 1순위 차별 기능. `cache_control: {type: 'ephemeral'}` 마커를 system prompt · tool 정의 위치에 부착하면 5분~1시간 캐시 → **입력 토큰 약 90% 할인**.

LangChain v1 에서는 `AnthropicPromptCachingMiddleware()` 를 `create_agent` 의 `middleware=[...]` 에 넣으면 자동으로 마커가 붙는다.

```python
from langchain.agents import create_agent
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware

agent = create_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[...],
    system_prompt=LONG_SYSTEM_PROMPT,
    middleware=[AnthropicPromptCachingMiddleware(ttl='1h', min_messages_to_cache=2)],
)
```

확인: 응답의 `usage_metadata.input_token_details.cache_read` 값이 두 번째 호출부터 0 이상으로 잡힌다.
상세는 [`11_provider_middleware/01_anthropic_prompt_caching.ipynb`](../11_provider_middleware/01_anthropic_prompt_caching.ipynb).

In [ ]:
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware
import inspect

# 미들웨어 파라미터를 한 번 시그니처로 확인 — 코드 작성 시 hallucination 방지
print(inspect.signature(AnthropicPromptCachingMiddleware))

## 13.01.5 가격 · 한도 · 리전

(요금은 자주 바뀌므로 콘솔/문서 기준으로 다시 확인하라. 단위는 USD / 1M tokens.)

| 모델 | 컨텍스트 | 입력 | 출력 | 캐시 read | 비고 |
|------|---------|-----|------|-----------|------|
| `claude-opus-4-7` | 200k (1M beta) | 비싼 편 | 매우 높음 | 1/10 | 최상급 추론 · 코드 |
| `claude-sonnet-4-6` | 200k (1M beta) | 중간 | 중간 | 1/10 | 일반 에이전트 기본값 |
| `claude-haiku-4-5` | 200k | 저렴 | 저렴 | 1/10 | 분류·라우팅·요약 |

리전:

- Direct API: 글로벌 단일 엔드포인트
- AWS Bedrock: `us-east-1`, `us-west-2`, `eu-central-1`, `ap-northeast-1` 등
- Google Vertex: `us-east5`, `europe-west1`, `asia-southeast1` 등

한도:

- Tier 별 RPM/TPM. 신규 계정은 Tier 1 — 분당 약 50 RPM. Tier 2/3 는 사용 이력에 따라 자동 승격.
- 1h TTL prompt caching 은 **beta** — 헤더 `anthropic-beta: extended-cache-ttl-2025-04-11` 자동 부착됨.

## 핵심 정리

- **모델 1순위**: 일반 에이전트 = Sonnet, 분류·라우팅 = Haiku, 최상급 추론 = Opus
- **공급자 차별 1순위**: prompt caching → 거의 모든 멀티턴 에이전트에 기본 적용
- **Bedrock 경유** Claude 는 같은 모델이지만 패키지/캐시 미들웨어가 다르다 (`BedrockPromptCachingMiddleware`)
- 임베딩·벡터스토어는 **다른 공급자**와 짝지어야 한다 (OpenAI/Voyage/Cohere 등)

## 다음

- [`01_chat_models/02_anthropic.ipynb`](../01_chat_models/02_anthropic.ipynb) — ChatAnthropic 깊은 사용
- [`11_provider_middleware/01_anthropic_prompt_caching.ipynb`](../11_provider_middleware/01_anthropic_prompt_caching.ipynb) — 캐시 적중 측정
- [`11_provider_middleware/02_claude_bash_tool.ipynb`](../11_provider_middleware/02_claude_bash_tool.ipynb) ~ `05_anthropic_file_search.ipynb` — Claude 네이티브 도구 4종

## 참고

- `langchain-anthropic` PyPI: https://pypi.org/project/langchain-anthropic/
- 공식 docs: https://docs.langchain.com/oss/python/integrations/providers/anthropic
- Anthropic 모델 카탈로그: https://docs.claude.com/en/docs/about-claude/models